# Baseline Model

In [2]:
import joblib
import mlflow
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('Online_Shoppers_Base')

<Experiment: artifact_location='/home/rabat/projects/behavior-engineering/artifacts/1', creation_time=1790795785035, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790795785035, lifecycle_stage='active', name='Online_Shoppers_Base', tags={}, trace_location=None, workspace='default'>

In [4]:
df = pd.read_csv('../data/online_shoppers_intention.csv', sep=',')
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [5]:
cols_to_drop = ['BounceRates', 'Weekend', 'Region', 'Browser', 'OperatingSystems']
df_filtered = df.drop(columns=cols_to_drop)

num_distorted = ['ProductRelated', 'ProductRelated_Duration', 'PageValues']
for col in num_distorted:
    df_filtered[col] = np.log1p(df_filtered[col])

df_dummies = pd.get_dummies(df_filtered, drop_first=True)

df_dummies.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 22 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Administrative                 12330 non-null  int64  
 1   Administrative_Duration        12330 non-null  float64
 2   Informational                  12330 non-null  int64  
 3   Informational_Duration         12330 non-null  float64
 4   ProductRelated                 12330 non-null  float64
 5   ProductRelated_Duration        12330 non-null  float64
 6   ExitRates                      12330 non-null  float64
 7   PageValues                     12330 non-null  float64
 8   SpecialDay                     12330 non-null  float64
 9   TrafficType                    12330 non-null  int64  
 10  Revenue                        12330 non-null  bool   
 11  Month_Dec                      12330 non-null  bool   
 12  Month_Feb                      12330 non-null  bool   
 1

In [ ]:
target = 'Revenue'
X = df_dummies.drop(columns=[target])
y = df_dummies[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
with mlflow.start_run(run_name='Logistic_Regression_Baseline'):
    clf = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    )
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    y_pred_proba = clf.predict_proba(X_test_scaled)[:, 1]

    accuracy = float(accuracy_score(y_test, y_pred))
    mlflow.log_metric('test_accuracy', accuracy)
    
    f1 = float(f1_score(y_test, y_pred))
    mlflow.log_metric('test_f1', f1)
    
    precision = float(precision_score(y_test, y_pred))
    mlflow.log_metric('test_precision', precision)
    
    recall = float(recall_score(y_test, y_pred))
    mlflow.log_metric('test_recall', recall)

    pr_auc = float(average_precision_score(y_test, y_pred_proba))
    mlflow.log_metric('test_pr_auc', pr_auc)

    
print(
    '\nTreino concluído.\n'
    f'PR-AUC: {pr_auc:.2f}\n'
    f'Accuracy: {accuracy:.2f}\n'
    f'Precision: {precision:.2f}\n'
    f'Recall: {recall:.2f}\n'
    f'F1: {f1:.2f}'
)


🏃 View run Logistic_Regression_Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/5641918faa774be3a63edd913670b466
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Treino concluído.
PR-AUC: 0.64
Accuracy: 0.87
Precision: 0.55
Recall: 0.77
F1: 0.64


In [8]:
joblib.dump(clf, '../models/baseline_model.joblib')

['../models/baseline_model.joblib']